# EDA — Telemarketing JYB Dataset

**Source**: aguado/telemarketing-jyb-dataset  
**Role**: Comparison — contact campaigns and responses, channel/approach comparison  

Run ETL first: `python -m datathon.etl.telemarketing`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
DATA = Path('../../data')

## 1. Shape and Schema

In [ ]:
df = pd.read_parquet(DATA / 'processed/telemarketing.parquet')
print(f'Shape: {df.shape}')
df.dtypes

In [ ]:
df.head()

## 2. Missing Values

In [ ]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

## 3. Target Distribution

In [ ]:
target_col = next((c for c in df.columns if c.lower() in ('y', 'subscribed', 'target', 'response')), None)
if target_col:
    fig, ax = plt.subplots()
    df[target_col].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(f'Target distribution ({target_col})')
    plt.tight_layout()
    plt.show()
    print(df[target_col].value_counts(normalize=True))

## 4. Numeric Feature Distributions

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
if target_col in num_cols:
    num_cols = [c for c in num_cols if c != target_col]
if num_cols:
    df[num_cols].hist(bins=30, figsize=(16, 10))
    plt.tight_layout()
    plt.show()

## 5. Categorical Feature Distributions

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    fig, ax = plt.subplots(figsize=(8, 3))
    df[col].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(col)
    plt.tight_layout()
    plt.show()

## 6. Correlation Heatmap (Numeric Features)

In [ ]:
if num_cols:
    fig, ax = plt.subplots(figsize=(12, 10))
    corr_cols = num_cols + ([target_col] if target_col and target_col in df.select_dtypes(include='number').columns else [])
    sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', ax=ax)
    plt.tight_layout()
    plt.show()

## 7. Conversion Rate by Categorical Feature

In [ ]:
if target_col:
    for col in cat_cols:
        rates = df.groupby(col)[target_col].mean().sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(8, 3))
        rates.plot(kind='bar', ax=ax)
        ax.set_title(f'Conversion rate by {col}')
        ax.set_ylabel(f'P({target_col}=1)')
        plt.tight_layout()
        plt.show()

## 8. Leakage Audit

In [ ]:
leakage_cols = ['duration']
present = [c for c in leakage_cols if c in df.columns]
if present:
    print(f'WARNING: leakage columns present: {present}')
else:
    print('OK: no leakage columns found in processed dataset')

## 9. Summary

| Metric | Value |
|--------|-------|
| Rows | — |
| Columns (post-ETL) | — |
| Conversion rate | — |
| Missing values | — |
| Leakage columns dropped | `duration` (if present) |

> Fill in after running notebook.